In [ ]:
import plotly.graph_objects as go
import seaborn as sns  # for a nice palette

def plot_sankey_model_to_instance_colored(experiments_data, top_n=None):
    import plotly.graph_objects as go
    import seaborn as sns

    # Collect and sort all instance IDs
    all_instance_ids = set()
    for v in experiments_data.values():
        all_instance_ids |= v['resolved_ids']
    sorted_instance_ids = sorted(all_instance_ids)

    if top_n:
        sorted_instance_ids = sorted_instance_ids[:top_n]

    model_names = list(experiments_data.keys())
    model_to_idx = {m: i for i, m in enumerate(model_names)}
    instance_to_idx = {iid: i + len(model_names) for i, iid in enumerate(sorted_instance_ids)}

    labels = model_names + sorted_instance_ids
    sources, targets, values, link_colors = [], [], [], []

    palette = sns.color_palette("tab10", len(model_names))
    model_colors = {
        m: f'rgba({int(r*255)},{int(g*255)},{int(b*255)},0.6)'
        for m, (r, g, b) in zip(model_names, palette)
    }

    for model, data in experiments_data.items():
        for iid in sorted(data['resolved_ids']):
            if iid in instance_to_idx:
                sources.append(model_to_idx[model])
                targets.append(instance_to_idx[iid])
                values.append(1)
                link_colors.append(model_colors[model])

    node_colors = [model_colors[m] for m in model_names] + ['rgba(220,220,220,0.4)'] * len(sorted_instance_ids)

    fig = go.Figure(go.Sankey(
        node=dict(
            pad=10,
            thickness=15,
            line=dict(color="black", width=0.5),
            label=labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors
        )
    ))

    fig.update_layout(
        title_text="Model → Instance ID Resolutions",
        font_size=10,
        height=800,
        width=1200
    )
    fig.write_html("sankey_model_to_instance.html")
    # fig.show()


def plot_sankey_model_to_repo_colored(experiments_data, all_instance_ids):
    import plotly.graph_objects as go
    import seaborn as sns

    instance_to_repo = {iid: iid.split('__')[0] for iid in all_instance_ids}
    model_names = list(experiments_data.keys())
    repo_names = sorted(set(instance_to_repo[iid] for iid in all_instance_ids))

    model_to_idx = {m: i for i, m in enumerate(model_names)}
    repo_to_idx = {r: i + len(model_names) for i, r in enumerate(repo_names)}
    labels = model_names + repo_names

    sources, targets, values, link_colors = [], [], [], []

    palette = sns.color_palette("tab10", len(model_names))
    model_colors = {
        m: f'rgba({int(r*255)},{int(g*255)},{int(b*255)},0.6)'
        for m, (r, g, b) in zip(model_names, palette)
    }

    for model, data in experiments_data.items():
        repo_counter = {}
        for iid in data['resolved_ids']:
            repo = instance_to_repo.get(iid)
            if repo:
                repo_counter[repo] = repo_counter.get(repo, 0) + 1

        for repo, count in repo_counter.items():
            sources.append(model_to_idx[model])
            targets.append(repo_to_idx[repo])
            values.append(count)
            link_colors.append(model_colors[model])

    node_colors = [model_colors[m] for m in model_names] + ['rgba(200,200,200,0.4)'] * len(repo_names)

    fig = go.Figure(go.Sankey(
        node=dict(
            pad=10,
            thickness=15,
            line=dict(color="black", width=0.5),
            label=labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors
        )
    ))

    fig.update_layout(
        title_text="Model → Repository Resolutions",
        font_size=10,
        height=800,
        width=1000
    )
    fig.write_html("sankey_model_to_repo.html")
    # fig.show()

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import load_dataset
import numpy as np
import random
from matplotlib.colors import ListedColormap

def load_json_file(file_path):
    """Load and parse a JSON file."""
    with open(file_path, 'r') as f:
        return json.load(f)

def get_experiment_name(file_path):
    """Extract experiment name from file path."""
    return os.path.basename(file_path).split('.')[0]

def analyze_swe_bench_results(file_paths):
    # Load SWE-bench Lite dataset to get all instance IDs
    dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
    
    # Extract and sort all instance IDs
    all_instance_ids = sorted([instance['instance_id'] for instance in dataset])
    print(f"Total instances: {len(all_instance_ids)}")
    
    # Create a dictionary to store results for each experiment
    experiments_data = {}
    
    # Process each experiment file
    for file_path in file_paths:
        experiment_name = get_experiment_name(file_path)
        data = load_json_file(file_path)
        
        # Calculate overall resolved rate
        resolved_rate = data['resolved_instances'] / data['total_instances']
        print(f"{experiment_name}: Overall resolved rate = {resolved_rate:.2%}")
        
        # Store the experiment data
        experiments_data[experiment_name] = {
            'resolved_rate': resolved_rate,
            'resolved_ids': set(data.get('resolved_ids', [])),
            'unresolved_ids': set(data.get('unresolved_ids', [])),
            'error_ids': set(data.get('error_ids', [])),
            'empty_patch_ids': set(data.get('empty_patch_ids', [])),
            'incomplete_ids': set(data.get('incomplete_ids', []))
        }
    
    # Create dataframes for visualization
    resolved_df = create_instance_dataframe(all_instance_ids, experiments_data, 'resolved')
    unresolved_df = create_instance_dataframe(all_instance_ids, experiments_data, 'unresolved')
    
    # Create a repo-based summary dataframe
    repo_summary = create_repo_summary(all_instance_ids, experiments_data)
    
    # Plot instance-based visualizations
    plot_stacked_instance_distributions(resolved_df, unresolved_df, list(experiments_data.keys()))
    
    # Plot repository-based visualizations
    plot_stacked_repo_distributions(resolved_df, unresolved_df, list(experiments_data.keys()))
    
    # Plot 100% stacked bar charts by repository
    plot_percentage_stacked_bars(resolved_df, unresolved_df, list(experiments_data.keys()))
    
    # Plot heatmaps
    plot_heatmaps(resolved_df, unresolved_df, list(experiments_data.keys()))

    plot_sankey_model_to_instance_colored(experiments_data)
    plot_sankey_model_to_repo_colored(experiments_data, all_instance_ids)

    
    # Print the repository summary table
    print("\nRepository-wise Resolved Percentages:")
    print(repo_summary)
    
    # Save the repository summary table to a CSV file
    repo_summary.to_csv('repo_resolved_percentages.csv')
    
    return experiments_data, repo_summary

def create_instance_dataframe(all_instance_ids, experiments_data, resolution_type):
    """Create a dataframe for the given resolution type."""
    # Initialize the dataframe with instance IDs
    df = pd.DataFrame(index=all_instance_ids)
    
    # Extract the repository and issue number from each instance ID
    df['repository'] = df.index.map(lambda x: x.split('__')[0])
    
    # For each experiment, add a column indicating whether each instance is resolved/unresolved
    for exp_name, exp_data in experiments_data.items():
        if resolution_type == 'resolved':
            df[exp_name] = df.index.isin(exp_data['resolved_ids']).astype(int)
        else:  # unresolved - includes all non-resolved instances
            # An instance is unresolved if it's in unresolved_ids, error_ids, empty_patch_ids, or incomplete_ids
            unresolved_set = (exp_data['unresolved_ids'] | 
                             exp_data['error_ids'] | 
                             exp_data['empty_patch_ids'] |
                             exp_data['incomplete_ids'])
            df[exp_name] = df.index.isin(unresolved_set).astype(int)
    
    return df

def create_repo_summary(all_instance_ids, experiments_data):
    """Create a summary dataframe of resolved percentages by repository."""
    # Create a mapping from instance_id to repository
    instance_to_repo = {instance_id: instance_id.split('__')[0] for instance_id in all_instance_ids}
    
    # Initialize dictionary to store counts
    repo_counts = {}
    
    # Count instances by repository
    for repo in set(instance_to_repo.values()):
        repo_instances = [i for i, r in instance_to_repo.items() if r == repo]
        repo_counts[repo] = len(repo_instances)
    
    # Initialize the summary dataframe
    summary_data = []
    
    # For each experiment, calculate resolved percentage by repository
    for exp_name, exp_data in experiments_data.items():
        repo_resolved = {}
        
        # Count resolved instances by repository
        for repo in repo_counts.keys():
            repo_instances = [i for i, r in instance_to_repo.items() if r == repo]
            resolved_count = sum(1 for i in repo_instances if i in exp_data['resolved_ids'])
            repo_resolved[repo] = (resolved_count, repo_counts[repo], resolved_count / repo_counts[repo] * 100)
        
        # Add to summary data
        for repo, (resolved, total, percentage) in repo_resolved.items():
            summary_data.append({
                'Experiment': exp_name,
                'Repository': repo,
                'Resolved': resolved,
                'Total': total,
                'Percentage': percentage
            })
    
    # Create dataframe and pivot for better display
    summary_df = pd.DataFrame(summary_data)
    pivot_df = summary_df.pivot(index='Repository', columns='Experiment', values='Percentage')
    
    # Add a total row
    total_row = {}
    for exp_name in pivot_df.columns:
        exp_data = next(data for name, data in experiments_data.items() if name == exp_name)
        total_row[exp_name] = exp_data['resolved_rate'] * 100
    
    pivot_df.loc['Total'] = pd.Series(total_row)
    
    # Format the percentages
    return pivot_df.round(2)

def plot_stacked_instance_distributions(resolved_df, unresolved_df, experiment_names):
    """Plot the distribution of resolved and unresolved instances with stacked bars."""
    # Set style
    plt.style.use('ggplot')
    
    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 16))
    
    # Prepare the data for RESOLVED instances
    resolved_plot_df = resolved_df.drop(columns=['repository'])
    
    # Sort the instances by the repository for better visualization
    resolved_plot_df['repo'] = resolved_df['repository']
    resolved_plot_df = resolved_plot_df.sort_values(by=['repo'])
    resolved_plot_df = resolved_plot_df.drop(columns=['repo'])
    
    # Plot the stacked bars - each instance is one bar, colored sections are experiments
    resolved_plot_df[experiment_names].plot(kind='bar', stacked=True, ax=ax1, 
                                  colormap=ListedColormap(sns.color_palette("viridis", len(experiment_names))))
    
    ax1.set_title('Distribution of Resolved Instances', fontsize=16)
    ax1.set_xlabel('', fontsize=12)  # No label on x-axis as requested
    ax1.set_ylabel('Number of Experiments Resolved', fontsize=14)
    ax1.legend(title='Experiment', fontsize=10, loc='upper right')
    
    # Remove x-tick labels as requested
    ax1.set_xticklabels([])
    ax1.set_xticks([])
    
    # Add a text annotation showing the total number of instances
    ax1.text(0.01, 0.95, f"Total Instances: {len(resolved_plot_df)}", 
             transform=ax1.transAxes, fontsize=12, verticalalignment='top')
    
    # Prepare the data for UNRESOLVED instances
    unresolved_plot_df = unresolved_df.drop(columns=['repository'])
    
    # Sort the instances by the repository for better visualization
    unresolved_plot_df['repo'] = unresolved_df['repository']
    unresolved_plot_df = unresolved_plot_df.sort_values(by=['repo'])
    unresolved_plot_df = unresolved_plot_df.drop(columns=['repo'])
    
    # Plot the stacked bars - each instance is one bar, colored sections are experiments
    unresolved_plot_df[experiment_names].plot(kind='bar', stacked=True, ax=ax2, 
                                           colormap=ListedColormap(sns.color_palette("plasma", len(experiment_names))))
    
    ax2.set_title('Distribution of Unresolved Instances', fontsize=16)
    ax2.set_xlabel('', fontsize=12)  # No label on x-axis as requested
    ax2.set_ylabel('Number of Experiments Unresolved', fontsize=14)
    ax2.legend(title='Experiment', fontsize=10, loc='upper right')
    
    # Remove x-tick labels as requested
    ax2.set_xticklabels([])
    ax2.set_xticks([])
    
    # Add a text annotation showing the total number of instances
    ax2.text(0.01, 0.95, f"Total Instances: {len(unresolved_plot_df)}", 
             transform=ax2.transAxes, fontsize=12, verticalalignment='top')
    
    # Adjust layout and save figure
    plt.tight_layout()
    plt.savefig('swe_bench_stacked_instances_analysis.png', dpi=300)
    plt.show()

def plot_stacked_repo_distributions(resolved_df, unresolved_df, experiment_names):
    """Plot the distribution of resolved and unresolved instances by repository with stacked bars."""
    # Set style
    plt.style.use('ggplot')
    
    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 16))
    
    # Get counts by repository for RESOLVED instances
    resolved_repo_counts = {}
    for repo in resolved_df['repository'].unique():
        repo_df = resolved_df[resolved_df['repository'] == repo]
        repo_counts = {}
        for exp in experiment_names:
            repo_counts[exp] = repo_df[exp].sum()
        resolved_repo_counts[repo] = repo_counts
    
    # Convert to DataFrame for plotting
    resolved_repo_df = pd.DataFrame(resolved_repo_counts).T
    
    # Sort repositories alphabetically
    resolved_repo_df = resolved_repo_df.sort_index()
    
    # Plot stacked bar chart for resolved instances
    resolved_repo_df.plot(kind='bar', stacked=True, ax=ax1, 
                         colormap=ListedColormap(sns.color_palette("viridis", len(experiment_names))))
    
    ax1.set_title('Repository-wise Distribution of Resolved Instances', fontsize=16)
    ax1.set_xlabel('Repository', fontsize=14)
    ax1.set_ylabel('Number of Resolved Instances', fontsize=14)
    ax1.legend(title='Experiment', fontsize=10, loc='upper right')
    
    # Get counts by repository for UNRESOLVED instances
    unresolved_repo_counts = {}
    for repo in unresolved_df['repository'].unique():
        repo_df = unresolved_df[unresolved_df['repository'] == repo]
        repo_counts = {}
        for exp in experiment_names:
            repo_counts[exp] = repo_df[exp].sum()
        unresolved_repo_counts[repo] = repo_counts
    
    # Convert to DataFrame for plotting
    unresolved_repo_df = pd.DataFrame(unresolved_repo_counts).T
    
    # Sort repositories alphabetically
    unresolved_repo_df = unresolved_repo_df.sort_index()
    
    # Plot stacked bar chart for unresolved instances
    unresolved_repo_df.plot(kind='bar', stacked=True, ax=ax2, 
                           colormap=ListedColormap(sns.color_palette("plasma", len(experiment_names))))
    
    ax2.set_title('Repository-wise Distribution of Unresolved Instances', fontsize=16)
    ax2.set_xlabel('Repository', fontsize=14)
    ax2.set_ylabel('Number of Unresolved Instances', fontsize=14)
    ax2.legend(title='Experiment', fontsize=10, loc='upper right')
    
    # Adjust layout and save figure
    plt.tight_layout()
    plt.savefig('swe_bench_repo_analysis.png', dpi=300)
    plt.show()
    
    # Export the repository data for further analysis
    resolved_repo_df.to_csv('resolved_by_repo.csv')
    unresolved_repo_df.to_csv('unresolved_by_repo.csv')
    print("Exported repository counts to CSV files")
    
    return resolved_repo_df, unresolved_repo_df

def plot_percentage_stacked_bars(resolved_df, unresolved_df, experiment_names):
    """Plot 100% stacked bar charts showing normalized resolved/unresolved distribution by repository."""
    # Set style for better visualization
    plt.figure(figsize=(16, 10))
    
    # Define consistent colors for each model
    # Use a consistent color mapping that ensures all models have distinct colors
    model_colors = {
        'claude-3-7-sonnet-latest': '#1f77b4',  # Blue
        'deepseek-reasoner': '#2ca02c',        # Green
        'gemini-2.5-pro-preview-05-06': '#9467bd',                 # Purple
        'gpt-4': '#ff69b4',                    # Pink
        'grok-3-latest': '#ffcb04',            # Yellow
        'mistral-large-latest': '#17becf'      # Light blue
    }
    
    # Make sure we have colors for all experiment names
    for model in experiment_names:
        if model not in model_colors:
            # Assign a random color if not already defined
            model_colors[model] = f'#{random.randint(0, 0xFFFFFF):06x}'
    
    # Get list of all repositories
    all_repos = sorted(resolved_df['repository'].unique())
    
    # Prepare data for resolved ratio
    resolved_data = {}
    
    for repo in all_repos:
        repo_instances = resolved_df[resolved_df['repository'] == repo].index
        total_instances = len(repo_instances)
        
        model_resolved = {}
        for model in experiment_names:
            if model in resolved_df.columns:
                resolved_count = resolved_df.loc[repo_instances, model].sum()
                model_resolved[model] = resolved_count / total_instances
            else:
                # Handle case where model might not be in the dataframe
                model_resolved[model] = 0
                
        resolved_data[repo] = model_resolved
    
    # Convert to DataFrame for plotting
    resolved_ratio_df = pd.DataFrame(resolved_data).T
    
    # Ensure all models are in the dataframe, even if they have zero values
    for model in experiment_names:
        if model not in resolved_ratio_df.columns:
            resolved_ratio_df[model] = 0
    
    # Plot the 100% stacked bar chart for resolved
    ax = resolved_ratio_df.plot(kind='bar', stacked=True, figsize=(16, 10), 
                             color=[model_colors[m] for m in resolved_ratio_df.columns],
                             width=0.8, edgecolor='white')
    
    # Set chart title and labels
    plt.title('Repository-wise Resolved Ratio (100% Stacked Bar Chart)', fontsize=20)
    plt.xlabel('Repository', fontsize=14)
    plt.ylabel('Resolved Proportion', fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Customize legend
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Adjust y-axis limit to match your example
    plt.ylim(0, 0.6)
    
    # Improve look and feel
    plt.tight_layout()
    plt.savefig('swe_bench_resolved_ratio.png', dpi=300)
    plt.show()
    
    # Now create the unresolved chart
    plt.figure(figsize=(16, 10))
    
    # Prepare data for unresolved ratio
    unresolved_data = {}
    
    for repo in all_repos:
        repo_instances = unresolved_df[unresolved_df['repository'] == repo].index
        total_instances = len(repo_instances)
        
        model_unresolved = {}
        for model in experiment_names:
            if model in unresolved_df.columns:
                unresolved_count = unresolved_df.loc[repo_instances, model].sum()
                model_unresolved[model] = unresolved_count / total_instances
            else:
                # Handle case where model might not be in the dataframe
                model_unresolved[model] = 0
                
        unresolved_data[repo] = model_unresolved
    
    # Convert to DataFrame for plotting
    unresolved_ratio_df = pd.DataFrame(unresolved_data).T
    
    # Ensure all models are in the dataframe, even if they have zero values
    for model in experiment_names:
        if model not in unresolved_ratio_df.columns:
            unresolved_ratio_df[model] = 0
    
    # Plot the 100% stacked bar chart for unresolved
    ax = unresolved_ratio_df.plot(kind='bar', stacked=True, figsize=(16, 10), 
                               color=[model_colors[m] for m in unresolved_ratio_df.columns],
                               width=0.8, edgecolor='white')
    
    # Set chart title and labels
    plt.title('Repository-wise Unresolved Ratio (100% Stacked Bar Chart)', fontsize=20)
    plt.xlabel('Repository', fontsize=14)
    plt.ylabel('Unresolved Proportion', fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Customize legend
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Adjust y-axis limit to match your example
    plt.ylim(0, 0.6)
    
    # Improve look and feel
    plt.tight_layout()
    plt.savefig('swe_bench_unresolved_ratio.png', dpi=300)
    plt.show()
    
    # Export the ratio data for further analysis
    resolved_ratio_df.to_csv('resolved_ratio_by_repo.csv')
    unresolved_ratio_df.to_csv('unresolved_ratio_by_repo.csv')
    print("Exported ratio data to CSV files")
    
    return resolved_ratio_df, unresolved_ratio_df

def plot_heatmaps(resolved_df, unresolved_df, experiment_names):
    """Create enhanced heatmaps showing repository vs experiment performance."""
    # Set style for better visualization
    plt.figure(figsize=(14, 12))
    
    # Prepare data for resolved ratio heatmap
    resolved_ratio_data = []
    
    for repo in sorted(resolved_df['repository'].unique()):
        repo_df = resolved_df[resolved_df['repository'] == repo]
        total_instances = len(repo_df)
        
        for exp in experiment_names:
            # Calculate percentage of instances resolved by this experiment
            resolved_count = repo_df[exp].sum()
            resolved_ratio = (resolved_count / total_instances) * 100
            
            resolved_ratio_data.append({
                'Repository': repo,
                'Experiment': exp,
                'Resolved Ratio (%)': resolved_ratio
            })
    
    # Convert to DataFrame for heatmap
    resolved_ratio_df = pd.DataFrame(resolved_ratio_data)
    resolved_heatmap_df = resolved_ratio_df.pivot(index='Repository', columns='Experiment', values='Resolved Ratio (%)')
    
    # Plot resolved ratio heatmap with improved styling
    plt.figure(figsize=(14, 12))
    ax = sns.heatmap(resolved_heatmap_df, annot=True, fmt=".1f", 
                    cmap="YlGnBu", linewidths=.5, 
                    cbar_kws={'label': 'Resolved Ratio (%)'})
    
    # Set chart title and labels
    plt.title('Model Performance by Repository (Resolved Ratio)', fontsize=18)
    plt.ylabel('Repository', fontsize=14)
    plt.xlabel('Model', fontsize=14)
    
    # Improve tick labels
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    
    # Adjust layout and save figure
    plt.tight_layout()
    plt.savefig('swe_bench_resolved_heatmap.png', dpi=300)
    plt.show()
    
    # Prepare data for unresolved ratio heatmap
    unresolved_ratio_data = []
    
    for repo in sorted(unresolved_df['repository'].unique()):
        repo_df = unresolved_df[unresolved_df['repository'] == repo]
        total_instances = len(repo_df)
        
        for exp in experiment_names:
            # Calculate percentage of instances unresolved by this experiment
            unresolved_count = repo_df[exp].sum()
            unresolved_ratio = (unresolved_count / total_instances) * 100
            
            unresolved_ratio_data.append({
                'Repository': repo,
                'Experiment': exp,
                'Unresolved Ratio (%)': unresolved_ratio
            })
    
    # Convert to DataFrame for heatmap
    unresolved_ratio_df = pd.DataFrame(unresolved_ratio_data)
    unresolved_heatmap_df = unresolved_ratio_df.pivot(index='Repository', columns='Experiment', values='Unresolved Ratio (%)')
    
    # Plot unresolved ratio heatmap with improved styling
    plt.figure(figsize=(14, 12))
    ax = sns.heatmap(unresolved_heatmap_df, annot=True, fmt=".1f", 
                    cmap="OrRd", linewidths=.5, 
                    cbar_kws={'label': 'Unresolved Ratio (%)'})
    
    # Set chart title and labels
    plt.title('Model Performance by Repository (Unresolved Ratio)', fontsize=18)
    plt.ylabel('Repository', fontsize=14)
    plt.xlabel('Model', fontsize=14)
    
    # Improve tick labels
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    
    # Adjust layout and save figure
    plt.tight_layout()
    plt.savefig('swe_bench_unresolved_heatmap.png', dpi=300)
    plt.show()
    
    # Export the heatmap data
    resolved_heatmap_df.to_csv('resolved_heatmap_data.csv')
    unresolved_heatmap_df.to_csv('unresolved_heatmap_data.csv')
    print("Exported heatmap data to CSV files")

In [ ]:
# file_paths = [
#     './claude-3-7-sonnet-latest.claude_groundtruth.json',
#     # './deepseek-chat.deepseek_chat_groundtruth.json', 
#     './deepseek-reasoner.deepseekr1_groundtruth.json', 
#     './gemini-2.5-pro-preview-03-25.gemini25_groundtruth.json', 
#     './gpt-4.1-2025-04-14.gpt_groundtruth.json', 
#     './grok-3-latest.grok3_groundtruth.json', 
#     './mistral-large-latest.mistral_groundtruth.json'
# ]


In [ ]:
# Paths, Accuracy, Average Paths, Resolved Rate
# Union / 4 voters   94.00%   3.67  31.00%
# Union / 3 voters + D   93.67%   2.99  31.67%
# Union / D   92.67%    2.87   32.33%
# Gemini / D   91.67%   2.67  32.33%

In [ ]:
# List of JSON file paths to analyze
file_paths = [
    
    # './gemini-2.5-pro-preview-03-25.union_4voters.json', 
    # './gemini-2.5-pro-preview-03-25.union_4voters_r1full.json', 
    # './gemini-2.5-pro-preview-03-25.union_deepseekr1full.json',
    # './gemini-2.5-pro-preview-03-25.gemini_deepseekr1full_concise.json', 

    'deepseek-reasoner.deepseekr1_concise_noreparse.json', 
    # 'deepseek-reasoner.deepseekr1_concise.json', 
    # 'gemini-2.5-pro-preview-05-06.union_4votegemini0506_union_4voters_r1full_reparse.json', 
    'gemini-2.5-pro-preview-05-06.gemini0506_reparse_groundtruth.json', 
    # 'gpt-4o-2024-08-06.gpt4o_concise_noreparse.json', 
    # 'gpt-4o-2024-08-06.gpt4o_concise.json', 
    'grok-3-beta.grok3_concise.json', 
    'claude-3-7-sonnet-latest.claude_concise.json'
]

# Run the analysis
results, repo_summary = analyze_swe_bench_results(file_paths)
